# 04 The accounting characteristics: four measures from the annual accounts

**What this notebook does.** For every member-month of the panel's window it finds the stock's company in Compustat through CRSP's link, picks the company's latest annual record that is at least six and at most 24 months old on the last trading day of month t-1, and measures four characteristics from it: book-to-market, earnings-to-price, operating profitability and asset growth. It counts by year how many member-months have each one, and why the others do not.

**Why it comes here.** The accounts of a fiscal year are published some weeks after it ends. A record used before it was published would give the models information no investor had. The six-month rule, from Gu, Kelly and Xiu (2020, p. 2248), leaves time for the accounts to appear.

**What it judges.** No gate and no expectation with a level. Gate G1 (b), judged in notebook 05, asks book-to-market and operating profitability to be present for at least 80 per cent of each month's members, and the other two for at least 90 per cent.

**The data.** CRSP and Compustat, from Wharton Research Data Services, accessed through Queen Mary University of London, extracted on 28 September 2026.

## Terms used in this notebook

| Term | Meaning |
|---|---|
| Compustat | the database of company accounts from S&P Global. Its annual fundamentals hold one record per company and fiscal year |
| GVKEY | Compustat's permanent number for a company |
| link | CRSP's table of which GVKEY belongs to which PERMNO, and over which dates. The study uses links of the types LC and LU with any primary flag, so that a second share class reaches its company |
| primary flag | CRSP's mark of which share class of a company is its main one: P or C for the main class, J or N for another |
| fiscal year end | the last day of a company's accounting year, Compustat's `datadate` |
| annual record | one company's accounts for one fiscal year |
| age of a record | the months from the month of its fiscal year end to month t-1. A record of December 2023 is 6 months old in June 2024 |
| book equity | stockholders' equity, plus deferred taxes and investment tax credit, minus preferred stock. Stockholders' equity is `seq`, else common equity, the item named `ceq`, plus `pstk`, else total assets minus total liabilities. Preferred stock is the redemption value `pstkrv`, else the liquidating value `pstkl`, else the par value `pstk`, else zero. Book equity that is not positive is missing |
| market value at the fiscal year end | the price at the fiscal year end times shares outstanding, `prcc_f` times `csho` |
| book-to-market | book equity, as Kenneth French's data library defines it, over the market value at the same fiscal year end, where the library divides by the market value at the end of December |
| earnings-to-price | income before extraordinary items, `ib`, over the same market value. Negative values are kept (Basu 1983) |
| operating profitability | revenue less cost of goods sold, selling, general and administrative expenses and interest expense, over book equity (Fama and French 2015). Revenue and at least one of the three costs must be present |
| asset growth | total assets over the same company's total assets a fiscal year before, less one (Cooper, Gulen and Schill 2008) |

## What I expect to see, and where it comes from

1. A GVKEY for at least 99.6 per cent of member-months in every year: Stock-Level Tracking's `sector_coverage_by_year.csv`, through the same links, finds one for at least 99.60 per cent of member-months in every year from 1990, the lowest in 2023.
2. A record of the right age for more than 97 per cent of the member-months with a GVKEY in every year. S&P 500 companies file accounts every year, so a record should be missing mainly for a company whose first fiscal year in Compustat ended less than six months before, such as a recent spin-off. A figure not read from the data.
3. A record without book-to-market for less than 10 per cent of member-months in every year. Most of these should have a book equity that is not positive, a share that grew after 2010 as large buybacks took the equity of some members below zero. The rest lack the price or the shares at the fiscal year end. A figure not read from the data.
4. Book-to-market and operating profitability present for more than 85 per cent of member-months in every year, earnings-to-price and asset growth for more than 95 per cent. Gate G1 (b) asks 80 and 90 per cent in every month.
5. In about half of the member-months the record used is 6 to 11 months old, and in the other half 12 to 17: a company with a December year end is read at 6 months in July and at 11 in the following June, and the record moves on each July.

Running time: about a minute.

In [ ]:
import hashlib
import json
import pathlib
import sys
import time

# The package lives in src/ beside this folder. The notebook finds the repository's root from where it runs.
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "src" / "mlrp").is_dir())
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd

from mlrp import constants as C
from mlrp import data as D
from mlrp.source import extraction as X
from mlrp.source import layer as L
from mlrp.source import returns as R
from mlrp.source import universe as U

# Stops with one sentence when the data root is not set, when the manifest is not the one the constants fix,
# or when one of the study's eight tables is not the one they record.
root, manifest = D.open_extraction()
print("manifest SHA-256", C.MANIFEST_SHA256[:12], "| files", manifest["counts"]["files"],
      "| rows", f"{manifest['counts']['rows']:,}", "| entries of the eight tables", C.MANIFEST_ENTRIES_SHA256[:12])

In [ ]:
g1a = D.read_result("g1a.json")
if not g1a["g1a_met"]:
    raise SystemExit("Gate G1 (a) is not met, so nothing after notebook 02 runs.")
print("Gate G1 (a): met, in notebook 02")

## 1. The member-months and the links

The universe is built again and cut to the panel's window. The link is read whole and cut to the two link types, with any primary flag. Each member-month takes the GVKEY of the link valid on the last trading day of month t-1, by the copied `link_gvkeys`.

In [ ]:
from mlrp import characteristics as K
universe = L.load_universe(root)
ulong = universe["ulong"]
inside = pd.PeriodIndex(ulong["month"], freq="M")
members = ulong[(inside >= pd.Period(C.PANEL_FIRST_MONTH, "M")) & (inside <= pd.Period(C.PANEL_LAST_MONTH, "M"))].reset_index(drop=True)
raw_links, bad_links = D.read_table(root, "crsp_ccm_lnkhist")
print(f"member-months {len(members):,} | link rows {len(raw_links):,}")

## 2. The annual records

The annual fundamentals are read with the company key, the fiscal year end, the fiscal year and the fifteen items the four characteristics need. `characteristics.annual_records` keeps one record per company and fiscal year end, computes book equity, the market value at the fiscal year end and operating profit, and attaches the total assets of the fiscal year before.

In [ ]:
t0 = time.time()
funda, bad_funda = D.read_table(root, "comp_funda")
records, record_counts = K.annual_records(funda)
print(f"rows {len(funda):,} | {time.time() - t0:.0f} s")
print(json.dumps(record_counts, indent=1))
print("date values present that did not parse:", {**bad_links, **bad_funda} or "none")

## 3. The four characteristics

`characteristics.accounting_characteristics` picks, for each member-month with a GVKEY, the record with the latest fiscal year end that is at least 6 months old on the last trading day of month t-1, and drops it when it is more than 24 months old.

**A worked example of the timing.** A company with a fiscal year ending on 31 December 2022 publishes its accounts in February or March 2023. For month t = June 2023, month t-1 is May 2023, 5 months after the year end, so the study still uses the record of December 2021, 17 months old. For month t = July 2023, month t-1 is June 2023, 6 months on, and the record of December 2022 is used. For a December year end that is the timing of Fama and French (1992), who match the accounts of a year with returns from July of the next.

In [ ]:
t0 = time.time()
accounting, more_than_one_link, _ = K.accounting_characteristics(members, raw_links, funda)
print(f"{time.time() - t0:.0f} s | member-months with more than one valid link: {more_than_one_link}")
print(accounting[list(K.ACCOUNTING)].notna().mean().round(4).to_string())

## 4. Coverage by year, and why a characteristic is missing

One table by year, an aggregate over each year's member-months: the share with a GVKEY, the share with a record of the right age, the share whose record gives no book-to-market (a book equity that is not positive, or no price or shares at the fiscal year end), the share with each characteristic, and the member-months that share their company with another member in the same month, the second share classes.

In [ ]:
a = accounting
year = pd.PeriodIndex(a["month"], freq="M").year
has_record = a["age_months"].notna()
shared = a[a["gvkey"].notna()].duplicated(["month", "gvkey"], keep=False)
table = pd.DataFrame({
    "member_months": a.groupby(year).size(),
    "share_with_gvkey": a["gvkey"].notna().groupby(year).mean(),
    "share_with_record": has_record.groupby(year).mean(),
    "share_with_record_without_book_to_market": (has_record & a["book_to_market"].isna()).groupby(year).mean(),
    **{f"share_with_{c}": a[c].notna().groupby(year).mean() for c in K.ACCOUNTING},
    "member_months_sharing_a_company": shared.reindex(a.index, fill_value=False).groupby(year).sum().astype("int64"),
}).round(6)
table.index.name = "year"
pd.set_option("display.width", 200)
print(table.to_string())
age = a["age_months"].dropna()
age_bands = pd.cut(age, bins=[5, 11, 17, 24], labels=["6 to 11", "12 to 17", "18 to 24"]).value_counts(normalize=True).sort_index().round(4)
print()
print("age of the record used, share of member-months with a record:")
print(age_bands.to_string())

## 5. The member-months without a company

The table above counts the member-months the links leave without a GVKEY. `characteristics.without_a_company` counts how many members they are, each member's member-months with its first and last year, and how many of them a link of another type would reach: a link in CRSP's table that names the member's PERMNO and is valid on the last trading day of month t-1, of a type other than LC and LU, with any primary flag. It prints counts and years only. Added on 24 November 2026.

In [ ]:
without = K.without_a_company(members, raw_links)
print(json.dumps(without, indent=1))

## 6. What is kept and what is committed

The four characteristics of every member-month, with its GVKEY and the age of its record, are kept in `local/accounting_characteristics.parquet` for notebook 05. Two files go into `results/20260928/`: the table by year and a summary with the counts of the annual records, the ages and the member-months without a company.

In [ ]:
D.write_local("accounting_characteristics.parquet", a.assign(month=a["month"].astype(str)))
written = [D.write_results("accounting_coverage_by_year.csv", frame=table.reset_index()),
           D.write_results("accounting_summary.json", record={
               "written_by": "notebooks/04_accounting_characteristics.ipynb", "manifest_sha256": C.MANIFEST_SHA256,
               "member_months": int(len(a)), "link_rows": int(len(raw_links)), "annual_records": record_counts,
               "member_months_with_more_than_one_valid_link": int(more_than_one_link),
               "age_of_the_record_used": {str(k): float(v) for k, v in age_bands.items()},
               "present": {c: int(a[c].notna().sum()) for c in K.ACCOUNTING},
               "date_values_not_parsed": {**bad_links, **bad_funda},
               "member_months_without_a_company": without})]
for path in written:
    print("written:", path.relative_to(ROOT))
results = pd.DataFrame([
    ("1. lowest yearly share with a GVKEY", "at least 0.996", f"{table['share_with_gvkey'].min():.4f}"),
    ("2. lowest yearly share with a record, of those with a GVKEY", "above 0.97", f"{(table['share_with_record'] / table['share_with_gvkey']).min():.4f}"),
    ("3. highest yearly share with a record and no book-to-market", "below 0.10", f"{table['share_with_record_without_book_to_market'].max():.4f}"),
    ("4. lowest yearly coverage of book-to-market and operating profitability", "above 0.85",
     f"{table[['share_with_book_to_market', 'share_with_operating_profitability']].min().min():.4f}"),
    ("4. lowest yearly coverage of earnings-to-price and asset growth", "above 0.95",
     f"{table[['share_with_earnings_to_price', 'share_with_asset_growth']].min().min():.4f}"),
    ("5. records 6 to 11 and 12 to 17 months old", "about half each", f"{age_bands.iloc[0]:.2f} and {age_bands.iloc[1]:.2f}"),
], columns=["expectation", "expected", "result"])
print(results.to_string(index=False))

## What was learned, and what could be wrong

- The table above sets each expectation beside its result. None has a verdict here: coverage is judged by gate G1 (b) in notebook 05.
- Compustat's records are the values as they stand in the extraction of 28 September 2026. A company that restated its accounts shows the restated figures, which were not known on the day the study uses them. The six-month rule protects against using a year's accounts before they were published. It does not undo restatements, and no data held here could.
- Two share classes of one company carry the same four values, since they have the same accounts. Their count by year is in the table.
- The 450 member-months without a company belong to five members: one in all 408 months, one in 39 months from 2021, and three in one month each. Links of two types the study does not use reach 447 of them: links of type LS reach all 447, and links of type LX reach 39 of the same 447. Three member-months are left without a company.
- A book equity that is not positive leaves book-to-market and operating profitability missing. Notebook 05 sets those missing values to the month's median after ranking, as for any missing value, and counts them.